In [7]:
# configs/interpreter.py
from pathlib import Path

from graphs import build_code_review_graph

REPO_ROOT: Path = Path.cwd().parents[2]
DEFAULT_SANDBOX_REPO: Path = REPO_ROOT / "out" / "sandbox"

DEFAULT_EXECUTE_TIMEOUT: int = 60
DEFAULT_MAX_OUTPUT_BYTES: int = 100_000
DEFAULT_INHERIT_ENV: bool = True
DEFAULT_VIRTUAL_MODE: bool = True

In [8]:
# prompts/agents.py
from prompts.template.builder import build_system_prompt

def build_interpreter_agent_prompt() -> str:
    """Build the P4 Sandbox/Interpreter agent's system prompt."""
    return build_system_prompt(
        assistant_name="Interpreter Agent",
        role="an autonomous computational and code execution assistant",
        domain="Python script execution, data analysis, and isolated computational workflows",
        mission=(
            "Execute computational, mathematical, and data analysis tasks in an isolated "
            "sandbox workspace. Write scripts, execute them, verify their outputs, and self-correct "
            "if errors occur."
        ),
        tools=(
            "You operate in a sandboxed workspace with full filesystem and shell access:\n"
            "- Use `write_file` to author complete Python scripts or data files.\n"
            "- Use `execute` to run commands (e.g. `python script.py`) and inspect stdout/stderr.\n"
            "- Use `edit_file` for targeted modifications or bug fixes in existing scripts.\n"
            "- Use `read_file`, `ls`, `glob`, and `grep` to inspect workspace contents and generated artifacts.\n"
            "- If a command fails or raises an error, analyze the traceback, fix the script, and re-execute."
        ),
        output_format=(
            "Provide clear, concise answers summarizing your findings, highlighting any generated "
            "artifacts, and including key figures directly computed by your code."
        ),
        constraints=(
            "- Never fabricate or guess computational results; always compute them using code.\n"
            "- Keep all execution and file manipulation strictly within your sandbox workspace.\n"
            "- Always verify script execution succeeds (exit code 0) before reporting final results."
        ),
    )

In [ ]:
# agents/interpreter.py
from langchain.agents.middleware import InterruptOnConfig
from typing import Any

from configs.interpreter import (
    DEFAULT_EXECUTE_TIMEOUT,
    DEFAULT_INHERIT_ENV,
    DEFAULT_MAX_OUTPUT_BYTES,
    DEFAULT_SANDBOX_DIR,
    DEFAULT_VIRTUAL_MODE,
)
from deepagents import create_deep_agent
from deepagents.backends import LocalShellBackend
from langgraph.checkpoint.memory import InMemorySaver
from models import llm_model
from prompts.agents import build_interpreter_agent_prompt


def make_interpreter_agent_in_jn(
    backend: LocalShellBackend | None = None,
    tier: str = "medium",
    checkpointer: Any | None = None,
    response_format: type | None = None,
    hitl: bool = True
):
    if backend is None:
        DEFAULT_SANDBOX_DIR.mkdir(parents=True, exist_ok=True)
        backend = LocalShellBackend(
            root_dir=DEFAULT_SANDBOX_DIR,
            virtual_mode=True,
            inherit_env=DEFAULT_INHERIT_ENV,
            timeout=DEFAULT_EXECUTE_TIMEOUT,
            max_output_bytes=DEFAULT_MAX_OUTPUT_BYTES,
        )

    interrupt_on: dict[str, bool | InterruptOnConfig] | None = (
    {"execute": True} if hitl else None
)

    
    return create_deep_agent(
        model=llm_model.get(tier),
        system_prompt=build_interpreter_agent_prompt(),
        backend=backend,
        checkpointer=checkpointer or InMemorySaver(),
        interrupt_on=interrupt_on
    )

In [ ]:
interpreter_agent_in_jn = make_interpreter_agent_in_jn()

In [11]:
# import dotenv
# dotenv.load_dotenv()
# from pathlib import Path
# from agents import make_interpreter_agent
# from configs.interpreter import DEFAULT_SANDBOX_DIR
# from utils.get_text import get_text

# # 1. Initialize agent with low tier for instant, cheap execution
# agent = make_interpreter_agent(tier='low')

# # 2. Invoke agent with a concrete task: create a file in sandbox
# response = agent.invoke(
#     {'messages': [{'role': 'user', 'content': '''ye dastane ghamangiz benevis ke shakhsiat haye un MIVE-HA bashan va vizhegi haye khasse har mive
#                    ra dar shakhsiat pardazie karakter haye dastan tasir bede. ba estefade az write_file natije ro dar file e dastan.text dar repo benevis '''}]},
#     config={'configurable': {'thread_id': 'exploration_run1'}}
# )

# # 3. Print verified agent response
# last_ai_msg = [m for m in response['messages'] if m.type == 'ai'][-1]
# print('=== Agent Output ===')
# print(get_text(last_ai_msg))

# # 4. Verify file on physical disk in sandbox
# # Note: DeepAgents maps virtual paths inside /workspace/ under root_dir
# created_file = DEFAULT_SANDBOX_DIR / 'workspace' / 'dastan.txt'
# print('\n=== Physical Disk Verification ===')
# print('Sandbox Path:', created_file)
# print('File Exists on Disk:', created_file.exists())
# if created_file.exists():
#     print('Content:', created_file.read_text().strip())

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage
from utils import get_text
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
agent = interpreter_agent_in_jn
config = {"configurable": {"thread_id": "interpreter_session_001"}}

In [14]:
def main():
    
    while True:
        try:
            user_msg = input("\nsandbox> ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nExiting session.")
            break
        
        if not user_msg:
            continue
        
        if user_msg in ["q", "quit", "exit"]:
            print("Session ended.")
            break
        
        result = agent.invoke(
            {
                "messages": [HumanMessage(content=user_msg)],
            },
            config=config,
        )
        print(f"\n{get_text(result['messages'][-1])}")
        
if __name__ == "__main__":
    main()

d:\Repos\lang_claude_exercise\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature, top_p will be ignored.
  request = self._build_request_config(
d:\Repos\lang_claude_exercise\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature, top_p will be ignored.
  request = self._build_request_config(
d:\Repos\lang_claude_exercise\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature, top_p will be ignored.
  request = self._build_request_config(
d:\Repos\lang_claude_exercise\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampli


Script `primes.py` neveshte va ejra shod. In ham code va natije-ye ejra:

```python
def is_prime(n):
    if n < 2:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True

def main():
    primes = [str(n) for n in range(1, 51) if is_prime(n)]
    
    with open("primes_list.txt", "w") as f:
        f.write(",".join(primes))
        
    print(f"Found {len(primes)} primes.")
    print(f"Largest prime: {primes[-1]}")

if __name__ == "__main__":
    main()
```

Natije-ye ejra:
- Tedad-e adad-e aval peyda shode: `15`
- Bozorgtarin adad-e aval: `47`
- Mohtaviyat-e file `primes_list.txt`: `2,3,5,7,11,13,17,19,23,29,31,37,41,43,47`
Session ended.


In [1]:
# interpreter_output_schema.py
from typing import Literal
from pydantic import BaseModel, Field

ArtifactType = Literal["chart", "markdown", "text", "json", "csv", "binary", "other"]

class ArtifactMetadata(BaseModel):
    name: str
    path: str
    type: ArtifactType
    desc: str
    size_bytes: int

class ExecutionManifest(BaseModel):
    summary: str
    artifacts: list[ArtifactMetadata]
    success: bool

In [3]:
def list_sandbox_artifacts(sandbox_dir: Path|None = None) -> list[ArtifactMetadata]:
    if sandbox_dir is None:
        sandbox_dir = DEFAULT_SANDBOX_DIR

    artifacts_dir = sandbox_dir / "artifacts"
    if not artifacts_dir.exists() or not artifacts_dir.is_dir():
        return []

    artifacts: list[ArtifactMetadata] = []
    for item in sorted(artifacts_dir.rglob("*")):
        if item.is_file():
            artifacts.append(
                ArtifactMetadata(
                    name=item.name,
                    path=item.relative_to(sandbox_dir).as_posix(),
                    size_bytes=item.stat().st_size,
                    type=_infer_artifact_type(item.suffix),
                    desc=
            )
        )
    return artifacts


In [4]:
def get_artifact_path(relative_path: str, sandbox_dir: Path|None = None) -> Path:
    if sandbox_dir is None:
        sandbox_dir = DEFAULT_SANDBOX_DIR

    artifacts_dir = (sandbox_dir / "artifacts").resolve()
    target = (sandbox_dir / relative_path).resolve()

    if not target.is_relative_to(artifacts_dir):
        msg = f"path traversal attempt: {relative_path} is outside artifact directory."
        raise ValueError(msg)
    return target

In [ ]:
def _infer_artifact_type(suffix: str) -> ArtifactType:
    normalized = suffix.lower()
    if normalized == ".csv":
        return "csv"
    if normalized == ".json":
        return "json"
    if normalized in {".png", ".jpg", ".jpeg", ".svg", ".pdf"}:
        return "chart"
    if normalized in {".md", ".markdown"}:
        return "markdown"
    if normalized in {".txt", ".log"}:
        return "text"
    return "other"

In [8]:
import time
from subagents.interpreter_subagent import make_interpreter_subagent
from configs.interpreter_config import DEFAULT_ARTIFACTS_DIR
from langchain_core.runnables import RunnableLambda
from langchain_core.messages import HumanMessage, AIMessage

DEFAULT_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
existing_file = DEFAULT_ARTIFACTS_DIR / "sales_report.csv"
existing_file.write_text("month,revenue\njan,1000\n", encoding="utf-8")

def sample_agent(inputs):
    time.sleep(0.05)

    with open(existing_file, "a", encoding="utf-8") as f:
        f.write("Feb,1500\n")

    chart_file = DEFAULT_ARTIFACTS_DIR / "revenue_growth.txt"
    chart_file.write_text("revenue grew by 50% in Feb", encoding="utf-8")

    return {
        "messages": [AIMessage(content="Financial analysis updated successfully")]
    }

subagent = make_interpreter_subagent(RunnableLambda(sample_agent))


In [9]:
result = subagent["runnable"].invoke({
    "messages": [HumanMessage(content="update sales and growth")]
})
print("=== Subagent Response Content ===")
print(result["messages"][0].content)

=== Subagent Response Content ===
Financial analysis updated successfully


### Deliverables Update:
- `[NEW]` `artifacts/revenue_growth.txt` (0.0 KB) [text] - TXT artifact (26 bytes)
- `[MODIFIED]` `artifacts/sales_report.csv` (0.0 KB) [csv] - CSV artifact (35 bytes)


In [ ]:
# Part I, Definition of Specification:
from typing import Literal
from pydantic import BaseModel, Field

class AgentManifest(BaseModel):
    name: str = Field(..., description="Unique slug for the agent")
    display_name: str
    description: str = Field(..., description="Exposed to Supervisor LLM for delegation")
    agent_type: Literal["pure", "supervisor"] = "pure"
    model_tier: Literal["low", "medium", "high"] = "medium"
    # If agent_type is "pure", list of tool names it can use
    tools: list[str] = Field(default_factory=list)
    # If agent_type is "supervisor", list of subagent names it can delegate to
    subagent_names: list[str] = Field(default_factory=list)



In [11]:
# part II, Definition of Registration:
from _collections_abc import Callable
from typing import Any

class EnterpriseAgentRegistry:
    def __init__(self) -> None:
        self._manifests = dict[str, AgentManifest] = {}
        self._factories = dict[str, Callable[..., Any]] = {}

    def register(self, manifest: AgentManifest):
        name = manifest.name
        def decorator(factory_fn: Callable[..., Any]):
            if name in self._manifests:
                raise ValueError(f"Agent {name} is already registered")
            self._manifests[name] = manifest
            self._factories[name] = factory_fn

            return factory_fn
        return decorator

    def get_manifest(self, name: str) -> AgentManifest:
        return self._manifests[name]
    def build_agents(self, name: str, **runtime_overrides: Any) -> Any:
        manifest = self._manifests[name]
        factory = self._factories[name]
        return factory(manifest, **runtime_overrides)

    def get_subagent_for_supervisor(self, names: list[str]) -> Callable[..., Any]:
        manifest = self._manifests[names]
        subagents = []
        for name in names:
            manifest = self.get_manifest(name)
            function = self.build_agents(name)


In [ ]:
import uuid
from typing import Any
from pydantic import BaseModel, Field
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.runnables import Runnable, RunnableLambda
from utils.get_text_util import get_text


class AgentManifest(BaseModel):
    name: str
    description: str
    tools: list[str] = Field(default_factory=list)


class EnterpriseAgentRegistry:
    def __init__(self) -> None:
        self._manifests: dict[str, AgentManifest] = {}
        self._factories: dict[str, Any] = {}

    def register(self, manifest: AgentManifest):
        def decorator(factory_fn):
            self._manifests[manifest.name] = manifest
            self._factories[manifest.name] = factory_fn
            return factory_fn
        return decorator

    def build_agent(self, name: str, **kwargs) -> Runnable:
        """Khorooji-e in function yek Runnable (CompiledGraph) hast."""
        manifest = self._manifests[name]
        factory = self._factories[name]
        return factory(manifest=manifest, **kwargs)

    # 💥 IN HAMOON FACTORY-E MAKHSOOSE CLOSURE HAST KE AGENT RO INJECT MIKONE:
    @staticmethod
    def _create_subagent_adapter(agent: Runnable):
        """Pure Dependency Injection:
        In function 'agent' ro be onvane parameter migire va yek closure-e
        kamalan safe va isolated misaze (mesle make_interpreter_subagent).
        """
        def adapter(state: dict[str, Any]) -> dict[str, Any]:
            task_text = get_text(state["messages"][-1])
            thread_id = str(uuid.uuid4())

            # Ejraye agent-e inject shode
            result = agent.invoke(
                {"messages": [HumanMessage(content=task_text)]},
                config={"configurable": {"thread_id": thread_id}},
            )

            output_text = get_text(result["messages"][-1])
            return {
                "messages": [AIMessage(content=output_text)],
                "thread_id": thread_id,
            }

        return adapter

    def get_subagents_for_supervisor(self, names: list[str]) -> list[dict]:
        subagents = []
        for name in names:
            manifest = self._manifests[name]

            # 1. Agent-e kham (Runnable) sakhte mishe
            agent_runnable = self.build_agent(name)

            # 2. Agent be function-e adapter inject mishe ta closure dorost bind beshe
            adapter_fn = self._create_subagent_adapter(agent=agent_runnable)

            # 3. Schema-ye DeepAgents wrap shode ba RunnableLambda
            subagents.append({
                "name": manifest.name,
                "description": manifest.description,
                "runnable": RunnableLambda(adapter_fn),
            })

        return subagents

In [ ]:
# coding_coordinator_app.py
"""application entry point interactive REPL loop for the P4 Phase C MultiAgent Coding Coordinator"""



In [ ]:
from dotenv import load_dotenv
load_dotenv()

import contextlib
import io
import sys

from agents.coding_coordinator_agent import coding_coordinator_agent, make_coding_coordinator_agent
from langchain_core.messages import HumanMessage
from utils.get_text_util import get_text
from utils.sandbox_util import list_sandbox_artifacts
from utils.session_util import resolve_session
from coordinator_config import DEFAULT_COORDINATOR_THREAD_ID

In [ ]:
default_thread_id = DEFAULT_COORDINATOR_THREAD_ID

In [ ]:
def main() -> None:
    """Launch the interactive REPL session for the Coding Coordinator."""

    raw_arg = sys.argv[1] if len(sys.argv) > 1 else None
    default_thread_id = DEFAULT_COORDINATOR_THREAD_ID
    session = resolve_session(
        raw_arg=raw_arg,
        default_thread_id=default_thread_id,
        default_agent=coding_coordinator_agent,
        agent_factory=make_coding_coordinator_agent(),
        app_name="coding_coordinator",
        cli_command_hint="python core/apps/coding_coordinator_app.py step[<index>]"
    )

    agent = session.agent
    thread_id = session.thread_id
    mode_name = session.mode_name
    is_secret = session.is_secret
    config = session.config

    if hasattr(sys.stdout, "reconfigure"):
        with contextlib.suppress(AttributeError, io.UnsupportedOperation):
            sys.stdout.reconfigure(encoding="utf-8")

    print("=" * 60)
    print("[Coordinator] P4 Phase C Multi-Agent Coding Coordinator")
    print("Orchestrates: code_interpreter (P4), code_review (P3), researcher(P1)")
    print("Sandbox Directory: out/sandbox/")
    print("Artifacts Directory: out/sandbox/artifacts/")
    if is_secret:
        print("Persistence: Secret Mode (In-Memory only — NOT saved to DB)")
    else:
        print("Persistence: SQLite checkpointer (checkpoints.db)")
    print(f"Session Mode: {mode_name}")
    print(f"Session Thread: {thread_id}")
    print("Type 'quit', 'exit' or 'q' to end session.")
    print("=" * 60)

    while True:

        try:
            user_input = input("\n[User]: ").strip()
        except (KeyboardInterrupt, EOFError):
            print("Session Terminated.")
            break

        if not user_input:
            continue

        if user_input in ["quit", "exit", "q"]:
            break


        print("\n[Coordinator is thinking & orchestrating subagents...]")
        try:
            result = agent.invoke(
                {"messages": [HumanMessage(content=user_input)]},
                config=config
            )
            output_text = get_text(result["messages"][-1])
        except Exception as e:
            print(f"\n[Error during execution]: {e}")


In [13]:
from graphs.code_review_parentgraph import build_code_review_graph
runnable  = RunnableLambda(build_code_review_graph())
type(x)

TypeError: Expected a callable type for `func`.Instead got an unsupported type: <class 'langgraph.graph.state.CompiledStateGraph'>